In [1]:
import time
from pop import Cds as cds, LiDAR, Util

lidar=LiDAR.Rplidar()

lidar.connect()
lidar.startMotor()
lasttime=0
dtime = time.time()
while time.time()-dtime<30:
    if time.time()-lasttime>0.03:
        data=lidar.getMap(size=(300,300))
        Util.imshow("map", data, width=600, height=600)
        lasttime=time.time()
lidar.stopMotor()

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [ ]:
lidar.startMotor()



In [ ]:
import time
from pop import Cds as cds, LiDAR, Util
from IPython.display import clear_output

lidar=LiDAR.Rplidar()

lidar.connect()
lidar.startMotor()
while True:
    try:
        vectors = lidar.getVectors()
        for v in vectors:
            print(v[0],v[1],v[2])
            clear_output()
    except KeyboardInterrupt:
        pass

lidar.stopMotor()
# vector [각도. 거리. 데이터신뢰도]
# 얼마나 데이터를 빠르게 내보내는가?
# 라이다의 속도가 1초가 몇번 회전 하는가?
# 한번 회전 할때 몇개의 데이터가 나오는가? 

In [2]:
import time
import numpy as np
from pop import LiDAR


def analyze_lidar(duration=10.0, quality_min=0, distance_min=0):
    """
    RPLIDAR 데이터 속도 분석 코드

    duration     : 측정 시간, 초
    quality_min  : 최소 품질값 필터
    distance_min : 최소 거리값 필터, mm
    """

    lidar = LiDAR.Rplidar()

    total_points = 0
    total_calls = 0
    rotation_count = 0

    points_in_current_rotation = 0
    points_per_rotation = []

    last_angle = None

    try:
        lidar.connect()
        lidar.startMotor()

        print("LiDAR 측정 시작")
        print(f"측정 시간: {duration}초")
        print("-" * 50)

        start_time = time.perf_counter()
        last_report_time = start_time

        while True:
            now = time.perf_counter()
            elapsed = now - start_time

            if elapsed >= duration:
                break

            vectors = lidar.getVectors()
            total_calls += 1

            if vectors is None or len(vectors) == 0:
                continue

            for v in vectors:
                angle = float(v[0])
                distance = float(v[1])
                quality = float(v[2])

                # 유효 데이터 필터
                if distance <= distance_min:
                    continue

                if quality < quality_min:
                    continue

                total_points += 1
                points_in_current_rotation += 1

                # 각도가 큰 값에서 작은 값으로 넘어가면 한 바퀴 회전했다고 판단
                # 예: 359도 -> 0도
                if last_angle is not None:
                    if last_angle > 300 and angle < 60:
                        rotation_count += 1
                        points_per_rotation.append(points_in_current_rotation)
                        points_in_current_rotation = 0

                last_angle = angle

            # 중간 출력
            if now - last_report_time >= 1.0:
                current_elapsed = now - start_time

                calls_per_sec = total_calls / current_elapsed
                points_per_sec = total_points / current_elapsed
                rotations_per_sec = rotation_count / current_elapsed
                rpm = rotations_per_sec * 60

                if len(points_per_rotation) > 0:
                    avg_points_per_rotation = np.mean(points_per_rotation)
                else:
                    avg_points_per_rotation = 0

                print(
                    f"[{current_elapsed:5.1f}s] "
                    f"호출/s={calls_per_sec:7.1f}, "
                    f"포인트/s={points_per_sec:8.1f}, "
                    f"회전/s={rotations_per_sec:5.2f}Hz, "
                    f"RPM={rpm:6.1f}, "
                    f"1회전 포인트={avg_points_per_rotation:7.1f}"
                )

                last_report_time = now

        total_elapsed = time.perf_counter() - start_time

        print("\n" + "=" * 50)
        print("최종 결과")
        print("=" * 50)

        print(f"총 측정 시간               : {total_elapsed:.2f} 초")
        print(f"getVectors() 총 호출 횟수   : {total_calls}")
        print(f"총 유효 포인트 개수         : {total_points}")
        print(f"감지한 회전 수              : {rotation_count}")

        if total_elapsed > 0:
            print(f"초당 getVectors() 호출 횟수 : {total_calls / total_elapsed:.2f} 회/s")
            print(f"초당 포인트 개수            : {total_points / total_elapsed:.2f} points/s")
            print(f"초당 회전 수                : {rotation_count / total_elapsed:.2f} Hz")
            print(f"분당 회전 수                : {(rotation_count / total_elapsed) * 60:.2f} RPM")

        if len(points_per_rotation) > 0:
            print(f"1회전당 평균 포인트 수      : {np.mean(points_per_rotation):.2f}")
            print(f"1회전당 최소 포인트 수      : {np.min(points_per_rotation)}")
            print(f"1회전당 최대 포인트 수      : {np.max(points_per_rotation)}")
            print(f"1회전당 표준편차            : {np.std(points_per_rotation):.2f}")
        else:
            print("회전이 감지되지 않았습니다.")

    except KeyboardInterrupt:
        print("\n사용자 중지")

    finally:
        try:
            lidar.stopMotor()
            print("LiDAR motor stopped")
        except Exception as e:
            print("stopMotor 처리 중 오류:", e)


def main():
    analyze_lidar(
        duration=10.0,
        quality_min=0,
        distance_min=0
    )


if __name__ == "__main__":
    main()

LiDAR 측정 시작
측정 시간: 10.0초
--------------------------------------------------
[  1.3s] 호출/s=    1.5, 포인트/s=   457.9, 회전/s= 0.74Hz, RPM=  44.7, 1회전 포인트=  269.0
[  2.4s] 호출/s=    5.9, 포인트/s=  1774.2, 회전/s= 5.45Hz, RPM= 326.8, 1회전 포인트=  304.2
[  3.4s] 호출/s=    8.0, 포인트/s=  2254.4, 회전/s= 7.66Hz, RPM= 459.5, 1회전 포인트=  284.7
[  4.4s] 호출/s=    9.2, 포인트/s=  2531.6, 회전/s= 9.01Hz, RPM= 540.9, 1회전 포인트=  274.2
[  5.5s] 호출/s=   10.1, 포인트/s=  2702.4, 회전/s= 9.87Hz, RPM= 592.5, 1회전 포인트=  268.8
[  6.5s] 호출/s=   10.6, 포인트/s=  2821.1, 회전/s=10.46Hz, RPM= 627.6, 1회전 포인트=  265.9
[  7.5s] 호출/s=   11.0, 포인트/s=  2898.7, 회전/s=10.89Hz, RPM= 653.2, 1회전 포인트=  263.3
[  8.6s] 호출/s=   11.3, 포인트/s=  2959.1, 회전/s=11.21Hz, RPM= 672.6, 1회전 포인트=  261.4
[  9.6s] 호출/s=   11.6, 포인트/s=  3002.0, 회전/s=11.46Hz, RPM= 687.8, 1회전 포인트=  259.7

최종 결과
총 측정 시간               : 10.04 초
getVectors() 총 호출 횟수   : 116
총 유효 포인트 개수         : 30043
감지한 회전 수              : 115
초당 getVectors() 호출 횟수 : 11.56 회/s
초당 포인트 개수            : 2992.98 points

In [4]:
import threading
import time

import numpy as np
from pop import LiDAR, Pilot


class FrontDistanceKeeper:
    def __init__(
        self,
        target_distance=700,      # 유지할 거리, mm
        tolerance=80,             # 허용 오차, mm
        front_range=10,           # 정면 기준 ±10도
        quality_min=10,
        min_valid_distance=100,
        max_valid_distance=3000,
        base_speed=25,
        max_speed=50,
        kp=0.04
    ):
        self.target_distance = target_distance
        self.tolerance = tolerance
        self.front_range = front_range
        self.quality_min = quality_min
        self.min_valid_distance = min_valid_distance
        self.max_valid_distance = max_valid_distance
        self.base_speed = base_speed
        self.max_speed = max_speed
        self.kp = kp

        self.lidar = LiDAR.Rplidar()
        self.car = Pilot.AutoCar()

        # 0도 ~ 359도까지 1도 간격 dictionary
        self.angle_dict = {degree: [] for degree in range(360)}

        self.front_distance = None
        self.front_count = 0

        self.lock = threading.Lock()
        self.running = threading.Event()

    def connect(self):
        self.lidar.connect()
        self.lidar.startMotor()

    def start(self):
        self.running.set()

        self.update_thread = threading.Thread(
            target=self.update,
            daemon=True
        )

        self.move_thread = threading.Thread(
            target=self.move,
            daemon=True
        )

        self.update_thread.start()
        self.move_thread.start()

    def stop(self):
        self.running.clear()

        try:
            self.car.stop()
        except Exception as e:
            print("car.stop() 오류:", e)

        try:
            self.lidar.stopMotor()
        except Exception as e:
            print("lidar.stopMotor() 오류:", e)

    def clear_angle_dict(self):
        for degree in range(360):
            self.angle_dict[degree].clear()

    def angle_to_degree(self, angle):
        """
        실수 각도를 1도 단위 정수 각도로 변환
        예:
        0.2  -> 0
        0.7  -> 1
        359.6 -> 0 으로 넘어갈 수 있으므로 % 360 처리
        """
        return int(round(angle)) % 360

    def get_front_degrees(self):
        """
        정면을 0도라고 가정한다.
        front_range=10이면
        350, 351, ..., 359, 0, 1, ..., 10
        """
        degrees = []

        for d in range(-self.front_range, self.front_range + 1):
            degrees.append(d % 360)

        return degrees

    def calc_front_average(self):
        """
        angle_dict에서 정면 ±10도 데이터만 모아서 평균 거리 계산
        """
        front_distances = []

        for degree in self.get_front_degrees():
            front_distances.extend(self.angle_dict[degree])

        if len(front_distances) == 0:
            return None, 0

        return float(np.mean(front_distances)), len(front_distances)

    def update(self):
        """
        LiDAR 데이터를 계속 읽어서
        1도 단위 dictionary에 저장하고,
        정면 ±10도 평균 거리를 계산한다.
        """
        while self.running.is_set():
            try:
                vectors = self.lidar.getVectors()

                # 이번 스캔 데이터를 새로 저장
                self.clear_angle_dict()

                for angle, distance, quality in vectors:
                    angle = float(angle)
                    distance = float(distance)
                    quality = float(quality)

                    if quality < self.quality_min:
                        continue

                    if distance < self.min_valid_distance:
                        continue

                    if distance > self.max_valid_distance:
                        continue

                    degree = self.angle_to_degree(angle)

                    # 1도 단위 dictionary에 거리 저장
                    self.angle_dict[degree].append(distance)

                front_avg, front_count = self.calc_front_average()

                with self.lock:
                    self.front_distance = front_avg
                    self.front_count = front_count

                time.sleep(0.01)

            except Exception as e:
                print("update 오류:", e)
                time.sleep(0.1)

    def calc_speed(self, error):
        speed = self.base_speed + abs(error) * self.kp
        speed = int(min(speed, self.max_speed))
        return speed

    def move(self):
        """
        정면 평균 거리 기준으로 자동차 이동
        """
        while self.running.is_set():
            with self.lock:
                distance = self.front_distance
                count = self.front_count

            if distance is None:
                self.car.stop()
                print("정면 데이터 없음 -> 정지")
                time.sleep(0.1)
                continue

            error = distance - self.target_distance
            speed = self.calc_speed(error)

            if abs(error) <= self.tolerance:
                self.car.stop()
                state = "STOP"

            elif error > 0:
                # 현재 거리가 목표보다 멀다 -> 전진
                self.car.forward(speed)
                state = "FORWARD"

            else:
                # 현재 거리가 목표보다 가깝다 -> 후진
                self.car.backward(speed)
                state = "BACKWARD"

            print(
                f"state={state:8s} | "
                f"front_avg={distance:7.1f} mm | "
                f"target={self.target_distance:4d} mm | "
                f"error={error:7.1f} | "
                f"speed={speed:3d} | "
                f"front_points={count}"
            )

            time.sleep(0.05)


def main():
    robot = FrontDistanceKeeper(
        target_distance=700,   # 70cm 유지
        tolerance=80,          # ±8cm 이내면 정지
        front_range=10,        # 정면 ±10도
        quality_min=10,
        min_valid_distance=100,
        max_valid_distance=3000,
        base_speed=40,
        max_speed=65,
        kp=0.04
    )

    try:
        robot.connect()
        robot.start()

        while True:
            time.sleep(1)

    except KeyboardInterrupt:
        print("\n사용자 중지")

    finally:
        robot.stop()
        print("프로그램 종료")


if __name__ == "__main__":
    main()

state=FORWARD  | front_avg= 1093.0 mm | target= 700 mm | error=  393.0 | speed= 55 | front_points=6
state=FORWARD  | front_avg= 1069.8 mm | target= 700 mm | error=  369.8 | speed= 54 | front_points=42
state=FORWARD  | front_avg= 1057.6 mm | target= 700 mm | error=  357.6 | speed= 54 | front_points=39
state=FORWARD  | front_avg= 1021.8 mm | target= 700 mm | error=  321.8 | speed= 52 | front_points=37
state=FORWARD  | front_avg=  956.2 mm | target= 700 mm | error=  256.2 | speed= 50 | front_points=37
state=FORWARD  | front_avg=  888.7 mm | target= 700 mm | error=  188.7 | speed= 47 | front_points=35
state=FORWARD  | front_avg=  822.7 mm | target= 700 mm | error=  122.7 | speed= 44 | front_points=35
state=STOP     | front_avg=  752.6 mm | target= 700 mm | error=   52.6 | speed= 42 | front_points=34
state=STOP     | front_avg=  697.9 mm | target= 700 mm | error=   -2.1 | speed= 40 | front_points=34
state=STOP     | front_avg=  698.8 mm | target= 700 mm | error=   -1.2 | speed= 40 | front_p